# Lab de Limpeza - Camada Silver: Vendas

**Comentário:** Descreva aqui as regras de negócio identificadas durante a limpeza manual.

In [1]:
import pandas as pd
import numpy as np
import s3fs

In [2]:
storage_options = {
    "key": "minio",
    "secret": "minio123",
    "client_kwargs": {"endpoint_url": "http://localhost:9000"}
}

In [3]:
fs = s3fs.S3FileSystem(**storage_options)
arquivos = fs.glob("bronze/vendas/**/*.csv")
arquivos

['bronze/vendas/ingestion_date=2026-04-29T00:30:04/vendas.csv']

In [4]:
df = pd.concat(
    [
        pd.read_csv(f"s3://{arquivo}", storage_options=storage_options) 
        for arquivo in arquivos
    ],
    ignore_index=True
)
df.head()

,id_venda,data_emissao,id_produto,id_cliente,qtd_caixas,preco_unitario,desconto_pct,representante,canal_venda,_ingested_at,_source_file
0,V000927,2024-02-28,P005,C0038,29,5.15,0.1464,Paulo Rodrigues,Varejo Tradicional,2026-04-29 00:30:04.665895,vendas.csv
1,V002651,2024-11-23,P015,C0103,19,7.58,0.1777,Carlos Mendes,Supermercado,2026-04-29 00:30:04.665895,vendas.csv
2,V001102,2024-07-20,P007,C0040,42,7.88,0.2234,Juliana Costa,ConveniÃªncia,2026-04-29 00:30:04.665895,vendas.csv
3,V001053,2024-04-01,P016,C0094,26,6.04,0.2038,Fernanda Souza,Varejo Tradicional,2026-04-29 00:30:04.665895,vendas.csv
4,V001872,2024-08-31,P006,C0002,8,4.23,0.0587,Rafael Lima,ConveniÃªncia,2026-04-29 00:30:04.665895,vendas.csv


Datas e Partições

In [8]:
df['data_emissao'] = df['data_emissao'].astype(str).str.strip()
df['data_emissao'] = pd.to_datetime(
    df['data_emissao'], 
    format='mixed', 
    dayfirst=True
)


In [9]:
df['ano'] = df['data_emissao'].dt.year
df['mes'] = df['data_emissao'].dt.month

Correção de Encoding

In [10]:
df['canal_venda'] = df['canal_venda'].str.encode('latin1').str.decode('utf-8', errors='ignore')

Cálculos de Receita

In [11]:
df['valor_bruto'] = df['qtd_caixas'] * df['preco_unitario']
df['valor_desconto'] = df['valor_bruto'] * df['desconto_pct']
df['valor_liquido'] = df['valor_bruto'] - df['valor_desconto']

Limpeza de Nulos críticos

In [12]:
df = df.dropna(subset=['ano', 'id_produto', 'id_cliente'])

In [13]:
try:
    df.to_parquet(
        path="s3://silver/venda/",
        partition_cols=['ano', 'mes'],
        storage_options=storage_options,
        index=False,
        compression="snappy"
    )
    print("✓ Vendas processadas e particionadas com sucesso!")
except Exception as e:
    print(f"✗ Erro ao salvar vendas: {e}")

✓ Vendas processadas e particionadas com sucesso!
